# Inferência — Transformer EN↔PT unificado (Keras 3 + TensorFlow)

Carrega o `melhor_modelo_unificado.keras` + `wordpiece_unificado.json` do treino e gera respostas com
**greedy**, **beam search** ou **amostragem**, com proteção contra repetição (n-gramas e penalidade).

1. Anexe o modelo (Kaggle: *Add Input → Models*) ou coloque os 3 arquivos na pasta do notebook.
2. Rode as células em ordem. O formato de entrada é o do treino: `[tarefa] instrução [SEP] entrada`.
3. Para avaliar, suba o `teste_unificado.csv` (7 MB, só o split de teste) e preencha `CSV_PATH`. Não precisa do dataset inteiro.

**Novo (v2):** no final há o **modo diálogo por turno** — você interpreta um personagem (X) e o modelo o outro (Y), com a narrativa e o histórico da conversa, como no dataset por turno. Também gera **narrativa a partir do título** e **título a partir da conversa** (útil para montar dataset sintético de conversas reais). Funciona com o modelo treinado com `dialogos_turnos.csv.gz`; um modelo treinado só com o dataset antigo responde, mas sem esse formato fica fraco.


In [ ]:
# ============================== PARÂMETROS ==============================
# Onde estão os arquivos do treino. Deixe None para procurar sozinho em /kaggle/input, na pasta atual e em /content.
MODELO_PATH    = None     # ex.: "/kaggle/input/models/.../melhor_modelo_unificado.keras"
TOKENIZER_PATH = None     # ex.: ".../wordpiece_unificado.json"
CONFIG_PATH    = None     # ex.: ".../config_unificado.json" (opcional)

# Para o bloco de AVALIAÇÃO (opcional): o mesmo CSV do treino. None = pula a avaliação.
CSV_PATH = None           # use o pequeno "teste_unificado.csv" (7 MB) ou "teste_amostra.csv" (0,4 MB) — NÃO precisa do dataset de 980 MB
CSV_E_SO_TESTE = True     # True = o CSV já contém só linhas de teste (usa todas). False = é o dataset completo: recria o split de teste
VAL_PROMILE, TEST_PROMILE = 10, 10   # só vale com CSV_E_SO_TESTE=False; tem que ser igual ao do treino

USAR_GPU = True          # se o kernel "morrer" na avaliação, ponha False (roda na CPU: mais lento, mas não derruba)

# Geração padrão
MODO = "beam"             # "greedy" | "beam" | "amostra"
MAX_NOVOS = 200           # máximo de tokens gerados
BEAMS = 4
ALPHA = 0.7               # penalidade de comprimento do beam (maior = respostas mais longas)
NAO_REPETIR_NGRAM = 6     # proíbe repetir qualquer sequência de N tokens (0 = desliga). N pequeno (2–3) quebra código/diálogos legítimos
PENALIDADE_REP = 1.0      # >1 desencoraja tokens já gerados (1.0 = desliga; valores >1 prejudicam código e texto longo)
TEMPERATURA, TOP_K, TOP_P = 0.8, 50, 0.95   # só para MODO="amostra"

# Precisão com que o modelo foi TREINADO (a política precisa ser a mesma ao carregar; senão dá erro de dtype float16/float32).
# Treino no Kaggle T4 = "mixed_float16". Se treinou com A100/TPU use "mixed_bfloat16"; se foi float32, "float32".
PRECISAO_TREINO = "mixed_float16"

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
if not USAR_GPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import re, json, glob, math, zlib, unicodedata, difflib, collections
import numpy as np
import tensorflow as tf
import keras
from keras import layers, ops

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

def achar(nome, dado):
    if dado:
        return dado
    for raiz in ("/kaggle/input", "/kaggle/working", "/content", ".", os.path.expanduser("~")):
        r = sorted(glob.glob(os.path.join(raiz, "**", nome), recursive=True))
        if r:
            return r[0]
    return None

# prefere o "melhor" modelo; senão o final; senão o último checkpoint
MODELO_PATH = MODELO_PATH or next((p for n in ("melhor_modelo_unificado.keras", "modelo_unificado.keras",
                                               "ultimo_modelo_unificado.keras") if (p := achar(n, None))), None)
TOKENIZER_PATH = achar("wordpiece_unificado.json", TOKENIZER_PATH)
CONFIG_PATH = achar("config_unificado.json", CONFIG_PATH)
print("modelo   :", MODELO_PATH); print("tokenizer:", TOKENIZER_PATH); print("config   :", CONFIG_PATH)
assert MODELO_PATH and TOKENIZER_PATH, "não achei o modelo/tokenizer: preencha MODELO_PATH e TOKENIZER_PATH acima"

In [ ]:
import tensorflow as tf
import keras
from keras import layers, ops


class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.position_embeddings = layers.Embedding(input_dim=sequence_length, output_dim=embed_dim)
        self.sequence_length, self.vocab_size, self.embed_dim = sequence_length, vocab_size, embed_dim
        self.dropout_rate = dropout_rate
        self.dropout = layers.Dropout(dropout_rate)

    def call(self, inputs):
        positions = ops.arange(0, ops.shape(inputs)[-1], 1)
        return self.dropout(self.token_embeddings(inputs) + self.position_embeddings(positions))

    def compute_mask(self, inputs, mask=None):
        return ops.not_equal(inputs, 0)

    def get_config(self):
        config = super().get_config()
        config.update({"sequence_length": self.sequence_length, "vocab_size": self.vocab_size,
                       "embed_dim": self.embed_dim, "dropout_rate": self.dropout_rate})
        return config


def _mha(num_heads, embed_dim, dropout_rate, attn_fp32):
    # attn_fp32=True calcula a atencao (e o softmax) em float32: mais estavel, usa mais memoria
    return layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads,
                                     dropout=dropout_rate, dtype="float32" if attn_fp32 else None)


class TransformerEncoder(layers.Layer):
    # pre_ln=True: normaliza ANTES de cada subcamada (estavel em float16). False = Post-LN original.
    def __init__(self, embed_dim, dense_dim, num_heads, dropout_rate=0.1, pre_ln=True, attn_fp32=False, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim, self.dense_dim, self.num_heads = embed_dim, dense_dim, num_heads
        self.dropout_rate, self.pre_ln, self.attn_fp32 = dropout_rate, pre_ln, attn_fp32
        self.attention = _mha(num_heads, embed_dim, dropout_rate, attn_fp32)
        self.dense_proj = keras.Sequential([
            layers.Dense(dense_dim, activation="relu"), layers.Dropout(dropout_rate), layers.Dense(embed_dim)])
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        pad = ops.cast(mask[:, None, :], dtype="int32") if mask is not None else None
        x = inputs
        if self.pre_ln:
            h = self.layernorm_1(x)
            x = x + ops.cast(self.attention(query=h, value=h, key=h, attention_mask=pad), x.dtype)
            return x + self.dense_proj(self.layernorm_2(x))
        a = self.attention(query=x, value=x, key=x, attention_mask=pad)
        x = self.layernorm_1(x + ops.cast(a, x.dtype))
        return self.layernorm_2(x + self.dense_proj(x))

    def get_config(self):
        config = super().get_config()
        config.update({"embed_dim": self.embed_dim, "dense_dim": self.dense_dim, "num_heads": self.num_heads,
                       "dropout_rate": self.dropout_rate, "pre_ln": self.pre_ln, "attn_fp32": self.attn_fp32})
        return config


class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, latent_dim, num_heads, dropout_rate=0.1, pre_ln=True, attn_fp32=False, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim, self.latent_dim, self.num_heads = embed_dim, latent_dim, num_heads
        self.dropout_rate, self.pre_ln, self.attn_fp32 = dropout_rate, pre_ln, attn_fp32
        self.attention_1 = _mha(num_heads, embed_dim, dropout_rate, attn_fp32)
        self.attention_2 = _mha(num_heads, embed_dim, dropout_rate, attn_fp32)
        self.dense_proj = keras.Sequential([
            layers.Dense(latent_dim, activation="relu"), layers.Dropout(dropout_rate), layers.Dense(embed_dim)])
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        x, enc = inputs
        x_mask, enc_mask = (None, None) if mask is None else mask
        n = ops.shape(x)[1]
        causal = ops.cast(ops.arange(n)[:, None] >= ops.arange(n), dtype="bool")[None, :, :]
        c = lambda t: ops.cast(t, x.dtype)
        if self.pre_ln:
            h = self.layernorm_1(x)
            x = x + c(self.attention_1(query=h, value=h, key=h, attention_mask=causal, query_mask=x_mask))
            h = self.layernorm_2(x)
            x = x + c(self.attention_2(query=h, value=enc, key=enc, query_mask=x_mask, key_mask=enc_mask))
            return x + self.dense_proj(self.layernorm_3(x))
        a1 = self.attention_1(query=x, value=x, key=x, attention_mask=causal, query_mask=x_mask)
        o1 = self.layernorm_1(x + c(a1))
        a2 = self.attention_2(query=o1, value=enc, key=enc, query_mask=x_mask, key_mask=enc_mask)
        o2 = self.layernorm_2(o1 + c(a2))
        return self.layernorm_3(o2 + self.dense_proj(o2))

    def compute_mask(self, inputs, mask=None):
        return mask[0] if mask is not None else None   # necessario para empilhar camadas

    def get_config(self):
        config = super().get_config()
        config.update({"embed_dim": self.embed_dim, "latent_dim": self.latent_dim, "num_heads": self.num_heads,
                       "dropout_rate": self.dropout_rate, "pre_ln": self.pre_ln, "attn_fp32": self.attn_fp32})
        return config

CUSTOM = {"PositionalEmbedding": PositionalEmbedding, "TransformerEncoder": TransformerEncoder,
          "TransformerDecoder": TransformerDecoder}

In [ ]:
from tokenizers import Tokenizer
tok = Tokenizer.from_file(TOKENIZER_PATH)
PAD, UNK, START, END, SEP = (tok.token_to_id(t) for t in ["[PAD]", "[UNK]", "[START]", "[END]", "[SEP]"])
VOCAB = tok.get_vocab_size()
ID2TOK = {i: t for t, i in tok.get_vocab().items()}
# as tags de tarefa foram adicionadas logo após os 5 especiais básicos (ids 5, 6, 7, ...): "[codigo]", "[instrucao]", ...
# (não dá para filtrar só pelo formato "[...]": o vocabulário também tem tokens comuns como "[]")
TAG_ID = {}
i = 5
while re.fullmatch(r"\[[a-z_]{3,}\]", ID2TOK.get(i, "")):
    TAG_ID[ID2TOK[i][1:-1]] = i
    i += 1
TAGS = sorted(TAG_ID)
TAG_IDS = set(TAG_ID.values())
print("vocabulário:", VOCAB, "| tarefas:", TAGS)

def _bytes_para_unicode():
    # mapa de bytes -> caracteres imprimiveis (o mesmo do ByteLevel do tokenizers)
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs:
            bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, [chr(c) for c in cs]))


BYTE_DECODER = {v: k for k, v in _bytes_para_unicode().items()}


def decode_ids(ids):
    # inverso exato do tokenizador: preserva quebras de linha, indentacao e acentos
    pecas = []
    for i in ids:
        i = int(i)
        if i == END:
            break
        if i in (PAD, START, SEP, UNK) or i in TAG_IDS:
            continue
        pecas.append(ID2TOK[i])
    s = "".join(p[2:] if p.startswith("##") else p for p in pecas)
    return bytes(BYTE_DECODER[c] for c in s if c in BYTE_DECODER).decode("utf-8", errors="replace")


def encode_prompt(tarefa, instrucao="", entrada="", seq_max=None):
    """[tarefa] instruction [SEP] input  — partes vazias são omitidas (igual ao treino)."""
    ids = [TAG_ID[tarefa]]
    feito = False
    for p in (instrucao, entrada):
        if p and p.strip():
            if feito:
                ids.append(SEP)
            ids += tok.encode(p).ids
            feito = True
    return ids[:seq_max] if seq_max else ids

assert decode_ids(tok.encode("def f(x):\n    return x  # olá, ação!").ids) == "def f(x):\n    return x  # olá, ação!"
print("round-trip do tokenizador ok")

In [ ]:
# ---------------- modelo ----------------
# as subcamadas (Dense/LayerNorm dentro dos blocos) são criadas com a política GLOBAL; ela tem que ser a do treino
keras.mixed_precision.set_global_policy(PRECISAO_TREINO)
modelo = keras.models.load_model(MODELO_PATH, custom_objects=CUSTOM, compile=False)
SEQ_MAX = modelo.get_layer("enc_embedding").sequence_length
print("parâmetros:", f"{modelo.count_params():,}", "| SEQ_MAX:", SEQ_MAX)

@tf.function(input_signature=[tf.TensorSpec([None, None], tf.int32), tf.TensorSpec([None, None], tf.int32)],
             autograph=False)
def _forward(e, d):
    return modelo({"encoder_inputs": e, "decoder_inputs": d}, training=False)

def probs_ultimo(enc, dec):
    """distribuição do PRÓXIMO token, (B, VOCAB), float32."""
    return _forward(tf.constant(enc, tf.int32), tf.constant(dec, tf.int32))[:, -1, :].numpy().astype("float64")

In [ ]:
# ---------------- geração: greedy / beam / amostragem, com anti-repetição ----------------
def _pad(seqs):
    L = max(len(s) for s in seqs)
    a = np.zeros((len(seqs), L), dtype="int32")
    for r, s in enumerate(seqs):
        a[r, :len(s)] = s
    return a

def _banidos_ngram(seq, n):
    """tokens que fechariam um n-grama já visto em `seq`."""
    if n <= 0 or len(seq) < n:
        return set()
    prefixo = tuple(seq[len(seq) - n + 1:]) if n > 1 else ()
    return {seq[i + n - 1] for i in range(len(seq) - n + 1) if tuple(seq[i:i + n - 1]) == prefixo}

def _ajusta(logp, seq, min_len_ok, nrep, pen):
    """aplica as regras ao log-prob (V,) da sequência `seq` (sem o START)."""
    logp = logp.copy()
    logp[PAD] = -1e30; logp[START] = -1e30; logp[SEP] = -1e30
    for t in TAG_IDS:
        logp[t] = -1e30
    if not min_len_ok:
        logp[END] = -1e30
    if pen and pen != 1.0 and seq:
        idx = np.fromiter(set(seq), dtype=int)
        logp[idx] = np.where(logp[idx] > 0, logp[idx] / pen, logp[idx] * pen)
    for b in _banidos_ngram(seq, nrep):
        logp[b] = -1e30
    return logp

def _amostra(logp, temp, top_k, top_p, rng):
    z = logp / max(temp, 1e-5)
    if top_k:
        corte = np.sort(z)[-min(top_k, len(z))]
        z = np.where(z < corte, -1e30, z)
    p = np.exp(z - z.max()); p /= p.sum()
    if top_p and top_p < 1.0:
        ordem = np.argsort(-p); c = np.cumsum(p[ordem])
        manter = ordem[: int(np.searchsorted(c, top_p)) + 1]
        q = np.zeros_like(p); q[manter] = p[manter]; p = q / q.sum()
    return int(rng.choice(len(p), p=p))

def _gerar_um_beam(enc_ids, max_novos, beams, alpha, nrep, pen, min_novos):
    enc = np.repeat(_pad([enc_ids]), beams, axis=0)
    seqs, scores = [[]], np.array([0.0])        # hipóteses vivas (sem START)
    prontas = []                                 # (score_normalizado, seq)
    for t in range(max_novos):
        dec = np.array([[START] + s for s in seqs], dtype="int32")
        probs = probs_ultimo(enc[:len(seqs)], dec)
        cand = []
        for k, s in enumerate(seqs):
            logp = _ajusta(np.log(np.maximum(probs[k], 1e-30)), s, t >= min_novos, nrep, pen)
            for tk_ in np.argpartition(-logp, beams)[:beams]:
                cand.append((scores[k] + logp[tk_], k, int(tk_)))
        cand.sort(key=lambda x: -x[0])
        novas, novos_sc = [], []
        for sc, k, tk_ in cand:
            if tk_ == END:
                prontas.append((sc / (((5 + len(seqs[k]) + 1) / 6) ** alpha), seqs[k]))
            else:
                novas.append(seqs[k] + [tk_]); novos_sc.append(sc)
            if len(novas) == beams:
                break
        seqs, scores = novas, np.array(novos_sc)
        # para quando a melhor pronta já supera o melhor que as vivas poderiam atingir
        if prontas and len(prontas) >= beams:
            melhor_viva = scores.max() / (((5 + len(seqs[0]) + 1) / 6) ** alpha) if len(seqs) else -1e30
            if max(p[0] for p in prontas) >= melhor_viva:
                break
        if not seqs:
            break
    if not prontas:
        prontas = [(s / (((5 + len(q)) / 6) ** alpha), q) for s, q in zip(scores, seqs)]
    return max(prontas, key=lambda x: x[0])[1]

def gerar(prompts, modo=None, max_novos=None, beams=None, alpha=None, nrep=None, pen=None,
          temperatura=None, top_k=None, top_p=None, min_novos=1, semente=0, tamanho_lote=16):
    """prompts: lista de (tarefa, instrucao, entrada). Retorna lista de textos."""
    modo = modo or MODO; max_novos = min(max_novos or MAX_NOVOS, SEQ_MAX - 1)
    beams = beams or BEAMS; alpha = ALPHA if alpha is None else alpha
    nrep = NAO_REPETIR_NGRAM if nrep is None else nrep; pen = PENALIDADE_REP if pen is None else pen
    temperatura = temperatura or TEMPERATURA; top_k = TOP_K if top_k is None else top_k
    top_p = TOP_P if top_p is None else top_p
    rng = np.random.default_rng(semente)
    encs = [encode_prompt(*p, seq_max=SEQ_MAX) for p in prompts]
    saidas = [None] * len(prompts)
    if modo == "beam":
        for i, e in enumerate(encs):
            saidas[i] = decode_ids(_gerar_um_beam(e, max_novos, beams, alpha, nrep, pen, min_novos))
        return saidas
    for ini in range(0, len(encs), tamanho_lote):          # greedy / amostra, em lotes
        idx = list(range(ini, min(ini + tamanho_lote, len(encs))))
        enc = _pad([encs[i] for i in idx])
        seqs = [[] for _ in idx]; fim = [False] * len(idx)
        for t in range(max_novos):
            # linhas vivas têm exatamente t tokens; as finalizadas são completadas com PAD (saída ignorada)
            dec = np.array([[START] + s + [PAD] * (t - len(s)) for s in seqs], dtype="int32")
            sai = _forward(tf.constant(enc), tf.constant(dec))[:, t, :].numpy().astype("float64")
            for r in range(len(idx)):
                if fim[r]:
                    continue
                logp = _ajusta(np.log(np.maximum(sai[r], 1e-30)), seqs[r], t >= min_novos, nrep, pen)
                nx = int(logp.argmax()) if modo == "greedy" else _amostra(logp, temperatura, top_k, top_p, rng)
                if nx == END:
                    fim[r] = True
                else:
                    seqs[r].append(nx)
            if all(fim):
                break
        for r, i in enumerate(idx):
            saidas[i] = decode_ids(seqs[r])
    return saidas

def perguntar(tarefa, instrucao="", entrada="", **kw):
    return gerar([(tarefa, instrucao, entrada)], **kw)[0]

In [ ]:
# ---------------- exemplos rápidos ----------------
# Formato: perguntar(tarefa, instrucao, entrada). Tarefas do modelo: ver a lista impressa acima.
print(perguntar("instrucao", "Explique em duas frases o que é um buraco negro."))
print("-" * 60)
print(perguntar("instrucao", "Traduza para o inglês:", "O gato dorme no sofá."))
print("-" * 60)
print(perguntar("codigo", "Escreva uma função Python que retorna o fatorial de n."))
print("-" * 60)
print(perguntar("perguntas_e_respostas", "Quem descobriu o Brasil?", "O Brasil foi alcançado pela frota de Pedro Álvares Cabral em 1500."))

In [ ]:
# ---------------- compara os modos de decodificação no mesmo prompt ----------------
prompt = ("instrucao", "Dê três dicas para estudar melhor.", "")
for modo in ("greedy", "beam", "amostra"):
    print(f"[{modo}]", perguntar(*prompt, modo=modo)[:400].replace("\n", " ⏎ "))
print("[greedy SEM anti-repetição]", perguntar(*prompt, modo="greedy", nrep=0, pen=1.0)[:400].replace("\n", " ⏎ "))

In [ ]:
# ---------------- avaliação no split de TESTE (opcional; precisa de CSV_PATH) ----------------
import pandas as pd
if not CSV_PATH:
    print("CSV_PATH vazio: avaliação pulada.")
else:
    def grupo(idv, fonte, inp):
        if "#" in idv:                              # linha derivada (turno/titulo): herda o grupo da conversa original
            idv = idv.split("#")[0]; fonte = idv.rsplit("-", 1)[0]
        if fonte.startswith("aya_"):
            return "y-" + inp
        if fonte in ("dialogo1", "dialogo1_pt"):
            return "d-" + idv.split("-")[-1]
        if fonte.startswith("assin"):
            return "a-" + inp
        return idv
    def divisao(chave):
        x = zlib.crc32(chave.encode("utf-8")) % 1000
        return "val" if x < VAL_PROMILE else ("test" if x < VAL_PROMILE + TEST_PROMILE else "train")
    COLS = ["id", "fonte", "tarefa", "instruction", "input", "output"]
    if CSV_E_SO_TESTE:
        test_df = pd.read_csv(CSV_PATH, usecols=COLS, dtype=str, keep_default_na=False)
    else:
        testes = []
        for ch in pd.read_csv(CSV_PATH, usecols=COLS, dtype=str, keep_default_na=False, chunksize=50000):
            m = [divisao(grupo(i, f, x)) == "test" for i, f, x in zip(ch["id"], ch["fonte"], ch["input"])]
            testes.append(ch[m])
        test_df = pd.concat(testes, ignore_index=True); del testes
    print(test_df.fonte.value_counts().to_string())

In [ ]:
N_POR_FONTE = 8           # amostras por fonte (beam é lento; use MODO="greedy" para avaliar rápido)
if not CSV_PATH or "test_df" not in globals():
    print("Nada a avaliar: defina CSV_PATH na célula de configuração e rode a célula anterior (que cria test_df).")
else:
    sim = lambda a, b: difflib.SequenceMatcher(None, a, b).ratio()
    linhas = []
    for fonte, g in test_df.groupby("fonte"):
        g = g.sample(min(N_POR_FONTE, len(g)), random_state=0)
        print("->", fonte, len(g), "linhas ...", flush=True)
        pred = gerar(list(zip(g["tarefa"], g["instruction"], g["input"])), modo="greedy", max_novos=100, tamanho_lote=4)
        ref = g["output"].tolist()
        em = np.mean([p.strip() == r.strip() for p, r in zip(pred, ref)])
        parecido = np.mean([sim(p, r) for p, r in zip(pred, ref)])
        linhas.append((fonte, len(g), em, parecido))
        r0 = g.iloc[0]
        print("=" * 70)
        print(f"[{fonte}] ENTRADA : {' | '.join(p for p in (r0['instruction'], r0['input']) if p)[:200]!r}")
        print(f"        MODELO  : {pred[0][:300]!r}")
        print(f"        ESPERADO: {ref[0][:300]!r}")
    print("\n%-16s %4s %12s %14s" % ("fonte", "n", "exact-match", "similaridade"))
    for f, n, em, sm in linhas:
        print("%-16s %4d %12.2f %14.2f" % (f, n, em, sm))

In [ ]:
# ---------------- conversa interativa ----------------
# Digite UMA linha no formato do treino:   [tarefa] instrução [SEP] entrada
#   - a tag [tarefa] é opcional (sem ela vale TAREFA_PADRAO); " [SEP] entrada" também é opcional
#   - exemplos:  [instrucao] Quanto é 1 + 1?
#                [dialogo_senso_comum] Maria estava feliz porque passou na prova. Ela chamou o amigo João.
#                [inferencia_textual] O cachorro passeia com a mulher [sep] A mulher passeia com o cachorro
#                [perguntas_e_respostas] Quem foi Dom Pedro I? [SEP] Dom Pedro I foi o primeiro imperador do Brasil.
# Comandos:  /tarefas   /exemplo [tarefa]   /modo greedy|beam|amostra   /max N   /sair   (linha vazia também sai)
TAREFA_PADRAO = "instrucao"

DICAS = {
    "instrucao": "[instrucao] Dê três dicas para estudar melhor.",
    "codigo": "[codigo] Escreva uma função Python que retorna o fatorial de n.",
    "dialogo_senso_comum": "[dialogo_senso_comum] Maria estava feliz porque passou na prova. Ela ligou para o amigo João para contar a novidade.",
    "inferencia_textual": "[inferencia_textual] O cachorro está sendo levado para passear pela mulher [sep] A mulher está passeando com o cachorro",
    "similaridade_semantica": "[similaridade_semantica] O cachorro está sendo levado para passear pela mulher [sep] A mulher está passeando com o cachorro",
    "perguntas_e_respostas": "[perguntas_e_respostas] Quem foi Dom Pedro I? [SEP] Dom Pedro I foi o primeiro imperador do Brasil.",
}

def interpretar(linha, tarefa_padrao=None):
    # transforma a linha digitada em (tarefa, instrucao, entrada); levanta ValueError com mensagem clara
    tarefa_padrao = tarefa_padrao or TAREFA_PADRAO
    m = re.match(r"^\s*\[([A-Za-z0-9_]+)\]\s*(.*)$", linha, re.S)
    if m and m.group(1) in TAG_ID:
        tarefa, resto = m.group(1), m.group(2)
    elif m and m.group(1).lower() != "sep":
        extra = " ('dialogo1_pt' é o nome de uma FONTE; a tarefa dele é dialogo_senso_comum)" if m.group(1).startswith("dialogo") else ""
        raise ValueError(f"tarefa desconhecida [{m.group(1)}]{extra}. Tarefas válidas: {TAGS}")
    else:
        tarefa, resto = tarefa_padrao, linha.strip()
    if " [SEP] " in resto:
        instrucao, entrada = resto.split(" [SEP] ", 1)
    else:
        instrucao, entrada = resto, ""
    if not (instrucao.strip() or entrada.strip()):
        raise ValueError("texto vazio depois da tag")
    return tarefa, instrucao, entrada

def montar_linha(tarefa, instrucao, entrada):
    partes = [p for p in (instrucao, entrada) if p and p.strip()]
    return f"[{tarefa}] " + " [SEP] ".join(partes)

def conversar():
    global MODO, MAX_NOVOS
    print("tarefas:", TAGS, "| modo:", MODO, "| /exemplo mostra uma linha pronta, /sair encerra")
    while True:
        linha = input("você> ").strip()
        if not linha or linha == "/sair":
            print("encerrado."); return
        if linha == "/tarefas":
            print(TAGS); continue
        if linha.startswith("/modo"):
            v = (linha.split() + [""])[1]
            if v in ("greedy", "beam", "amostra"):
                MODO = v; print("modo:", MODO)
            else:
                print("use: /modo greedy | beam | amostra")
            continue
        if linha.startswith("/max"):
            try:
                MAX_NOVOS = int(linha.split()[1]); print("max novos tokens:", MAX_NOVOS)
            except Exception:
                print("use: /max 200")
            continue
        if linha.startswith("/exemplo"):
            t = (linha.split() + [TAREFA_PADRAO])[1]
            if "test_df" in globals() and t in set(test_df["tarefa"]):
                r = test_df[test_df["tarefa"] == t].sample(1).iloc[0]
                print("copie e cole:\n" + montar_linha(r["tarefa"], r["instruction"], r["input"]) + f"\n(resposta esperada: {r['output'][:200]!r})")
            else:
                print("copie e cole:\n" + DICAS.get(t, f"[{t}] ..."))
            continue
        try:
            tarefa, instr, entrada = interpretar(linha)
        except ValueError as e:
            print("⚠️", e); continue
        print("→", perguntar(tarefa, instr, entrada), "\n")

In [ ]:
conversar()      # digite /exemplo para ver uma linha pronta de cada tarefa

## Modo diálogo por turno
Você é X e o modelo é Y (ou ao contrário com `/trocar`); o modelo recebe a narrativa e o histórico a cada fala.

In [ ]:
# ============================ MODO DIÁLOGO POR TURNO ============================
# Formato do treino (dataset por turno):
#   [dialogo_senso_comum] Próxima fala de {FALANTE}. [SEP] Narrativa: {narrativa}\nConversa:\n{Nome: fala\n...}   (ou "(início)")
#   [instrucao] Escreva uma narrativa curta a partir do título. [SEP] {título}
#   [instrucao] Escreva um título curto para esta conversa. [SEP] {Nome: fala\n...}
TXT = {
    "pt": dict(turno="Próxima fala de {f}.", narr="Narrativa", conv="Conversa", inicio="(início)",
               t2n="Escreva uma narrativa curta a partir do título.", c2t="Escreva um título curto para esta conversa."),
    "en": dict(turno="Next line by {f}.", narr="Narrative", conv="Conversation", inicio="(start)",
               t2n="Write a short narrative from the title.", c2t="Write a short title for this conversation."),
}
# geração dos diálogos: amostragem dá falas menos repetitivas que beam/greedy (mude para "beam" se preferir)
DIALOGO_KW = dict(modo="amostra", temperatura=0.8, top_k=40, top_p=0.9, nrep=6, max_novos=120, min_novos=2)


def _limpa(texto, falante=None):
    t = texto.strip().split("\n")[0].strip()          # uma fala só
    if falante and t.lower().startswith(falante.lower() + ":"):
        t = t[len(falante) + 1:].strip()
    return t


def proxima_fala(narrativa, historico, falante, idioma="pt", **kw):
    """historico: lista de (nome, fala). Retorna a próxima fala de `falante` (só o texto)."""
    T = TXT[idioma]
    hist = "\n".join(f"{n}: {f}" for n, f in historico) or T["inicio"]
    entrada = f"{T['narr']}: {narrativa.strip()}\n{T['conv']}:\n{hist}"
    saida = perguntar("dialogo_senso_comum", T["turno"].format(f=falante), entrada, **{**DIALOGO_KW, **kw})
    return _limpa(saida, falante)


def narrativa_do_titulo(titulo, idioma="pt", **kw):
    return perguntar("instrucao", TXT[idioma]["t2n"], titulo.strip(),
                     **{**DIALOGO_KW, "max_novos": 160, "min_novos": 8, **kw}).strip()


def titulo_da_conversa(historico, idioma="pt", **kw):
    """historico: lista de (nome, fala) OU texto cru 'Nome: fala' por linha. Retorna um título curto."""
    if isinstance(historico, str):
        texto = historico.strip()
    else:
        texto = "\n".join(f"{n}: {f}" for n, f in historico)
    return perguntar("instrucao", TXT[idioma]["c2t"], texto,
                     **{**DIALOGO_KW, "modo": "greedy", "max_novos": 24, **kw}).strip().split("\n")[0]


def dialogo_turnos(narrativa, eu, modelo, idioma="pt", comeca="eu", historico=None, **kw):
    """Você fala como `eu`, o modelo como `modelo`. Os dois nunca falam ao mesmo tempo; o histórico entra em cada turno.
    Comandos: /trocar (inverte os papéis: você vira o outro)  /desfazer  /historico  /titulo  /narrativa  /sair
    Linha vazia = o modelo fala de novo (útil para ver os dois lados)."""
    hist = list(historico or [])
    eu, modelo = eu.strip(), modelo.strip()
    vez = eu if comeca == "eu" else modelo
    print(f"Narrativa: {narrativa}\nVocê = {eu} | modelo = {modelo} | /trocar /desfazer /historico /titulo /narrativa /sair\n")
    while True:
        if vez == modelo:
            fala = proxima_fala(narrativa, hist, modelo, idioma, **kw)
            print(f"{modelo}> {fala}")
            hist.append((modelo, fala)); vez = eu; continue
        linha = input(f"{eu}> ").strip()
        if linha == "/sair":
            print("encerrado."); return hist
        if linha == "/historico":
            print("\n".join(f"{n}: {f}" for n, f in hist) or "(vazio)"); continue
        if linha == "/desfazer":
            if hist: print("removido:", hist.pop())
            continue
        if linha == "/trocar":
            eu, modelo = modelo, eu; print(f"agora você = {eu} | modelo = {modelo}"); vez = modelo if (hist and hist[-1][0] == eu) else eu
            continue
        if linha == "/titulo":
            print("título:", titulo_da_conversa(hist, idioma) if hist else "(converse um pouco antes)"); continue
        if linha == "/narrativa":
            print("narrativa:", narrativa); continue
        if not linha:
            vez = modelo; continue
        hist.append((eu, linha)); vez = modelo


def conversa_a_partir_do_titulo(titulo, eu, modelo, idioma="pt", **kw):
    """Gera a narrativa a partir do título e abre o diálogo com ela."""
    n = narrativa_do_titulo(titulo, idioma)
    print("Título:", titulo); return dialogo_turnos(n, eu, modelo, idioma, **kw)

In [ ]:
# ---------------- exemplos do modo por turno ----------------
narr = "Hiro ficou nervoso por contar seu segredo a Everly, mas decidiu que ela era confiável. Everly sempre foi uma boa amiga."

# 1) uma fala só
print(proxima_fala(narr, [("Everly", "Hiro, o que foi?")], "Hiro"))

# 2) título -> narrativa
print(narrativa_do_titulo("Alfred aceita um segundo emprego"))

# 3) conversa -> título (pode ser texto cru de voluntários, uma fala por linha "Nome: fala")
print(titulo_da_conversa("Ana: Oi, vim comprar lã.\nLoja: Temos vários tipos de fio.\nAna: Quero começar a tricotar."))

# 4) modelo contra ele mesmo: duas personagens, uma de cada vez
hist = []
for i in range(6):
    nome = "Everly" if i % 2 == 0 else "Hiro"
    hist.append((nome, proxima_fala(narr, hist, nome)))
    print(f"{nome}: {hist[-1][1]}")

In [ ]:
# ---------------- interativo: VOCÊ é uma personagem, o modelo é a outra ----------------
# Troque os nomes/narrativa. comeca="eu" -> você fala primeiro; "modelo" -> ele começa. /trocar inverte os papéis no meio da conversa.
hist = dialogo_turnos(narr, eu="Everly", modelo="Hiro", comeca="modelo")
# depois: print(titulo_da_conversa(hist))   # título para a conversa que você acabou de ter

## Base de conhecimento (busca antes de responder)
Indexa seus textos, **busca os trechos mais relevantes (BM25)** e os passa como contexto para `[perguntas_e_respostas] pergunta [SEP] contexto`. Sem dependências extras.
Limites: o modelo foi treinado com pouco Q&A *com contexto*; ele pode ignorar ou misturar o contexto. Confira sempre `contexto usado:` na saída.

In [ ]:
# ============================ BASE DE CONHECIMENTO (busca + resposta) ============================
# Fluxo: documentos -> trechos -> BM25 (busca por palavras) -> os melhores trechos viram o CONTEXTO da pergunta:
#   [perguntas_e_respostas] {pergunta} [SEP] {trechos encontrados}
# Sem dependências extras (BM25 em numpy). Coloque seus textos em .txt/.md numa pasta, ou passe uma lista de strings.
import os, re, glob, unicodedata, math, collections
import numpy as np

STOP = set("a o as os um uma uns umas de do da dos das em no na nos nas por para com sem sobre entre e ou que se ao aos à às "
           "é são foi era ser ter tem há como mais menos já não sim seu sua seus suas ele ela eles elas isso isto esse essa este esta "
           "qual quais quem quando onde porque the of and to in is".split())

def _norm(s):
    s = unicodedata.normalize("NFD", s.lower())
    return "".join(c for c in s if unicodedata.category(c) != "Mn")

def _tokens(s):
    return [w for w in re.findall(r"[a-z0-9]+", _norm(s)) if w not in STOP and len(w) > 1]

def dividir_em_trechos(texto, palavras=90, sobreposicao=20):
    """junta parágrafos/frases em trechos de ~`palavras` palavras, com sobreposição para não cortar uma ideia no meio."""
    frases = [f.strip() for f in re.split(r"(?<=[.!?])\s+|\n{2,}", texto) if f.strip()]
    trechos, atual = [], []
    for f in frases:
        atual.append(f)
        if sum(len(x.split()) for x in atual) >= palavras:
            trechos.append(" ".join(atual))
            cauda, n = [], 0
            for x in reversed(atual):                       # repete as últimas frases no próximo trecho
                n += len(x.split()); cauda.insert(0, x)
                if n >= sobreposicao: break
            atual = cauda if n < palavras else []
    if atual and (not trechos or " ".join(atual) != trechos[-1]):
        trechos.append(" ".join(atual))
    return trechos

class BaseConhecimento:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.trechos, self.fontes, self._tf, self._df, self._len = [], [], [], collections.Counter(), []

    def adicionar(self, texto, fonte="texto"):
        for t in dividir_em_trechos(texto):
            tf = collections.Counter(_tokens(t))
            if not tf: continue
            self.trechos.append(t); self.fontes.append(fonte); self._tf.append(tf)
            self._len.append(sum(tf.values())); self._df.update(tf.keys())
        return self

    def carregar_pasta(self, pasta, padroes=("*.txt", "*.md")):
        for pad in padroes:
            for caminho in sorted(glob.glob(os.path.join(pasta, "**", pad), recursive=True)):
                with open(caminho, encoding="utf-8", errors="ignore") as f:
                    self.adicionar(f.read(), fonte=os.path.relpath(caminho, pasta))
        print(f"{len(self.trechos)} trechos indexados")
        return self

    def buscar(self, pergunta, k=3):
        q = _tokens(pergunta)
        if not q or not self.trechos: return []
        N, avg = len(self.trechos), (sum(self._len) / len(self._len))
        scores = np.zeros(N)
        for w in set(q):
            df = self._df.get(w, 0)
            if not df: continue
            idf = math.log(1 + (N - df + 0.5) / (df + 0.5))
            for i, tf in enumerate(self._tf):
                f = tf.get(w, 0)
                if f: scores[i] += idf * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self._len[i] / avg))
        top = [int(i) for i in np.argsort(-scores)[:k] if scores[i] > 0]
        return [(self.trechos[i], self.fontes[i], float(scores[i])) for i in top]

def responder_com_base(base, pergunta, k=2, max_tokens_contexto=350, mostrar=True, **kw):
    """busca na base, monta o contexto (cabendo no limite do modelo) e pergunta ao modelo."""
    achados = base.buscar(pergunta, k=k)
    if not achados:
        return "(nada encontrado na base)"
    contexto, usados = "", []
    for texto, fonte, sc in achados:
        cand = (contexto + " " + texto).strip()
        if len(tok.encode(cand).ids) > max_tokens_contexto and contexto:
            break
        contexto = cand; usados.append((fonte, sc))
    if mostrar:
        print("contexto usado:", [(f, round(s, 1)) for f, s in usados]); print("-" * 60); print(contexto[:600], "..." if len(contexto) > 600 else ""); print("-" * 60)
    return perguntar("perguntas_e_respostas", pergunta, contexto, **{"modo": "greedy", "max_novos": 80, **kw})

In [ ]:
# ---------------- exemplo de uso ----------------
base = BaseConhecimento()
# a) de uma pasta com .txt / .md:   base.carregar_pasta("/caminho/da/pasta")
# b) ou direto de strings:
base.adicionar("O Brasil foi alcançado pela frota de Pedro Álvares Cabral em 22 de abril de 1500. A expedição partiu de Lisboa.", "historia")
base.adicionar("O Rio Amazonas é o maior rio do mundo em volume de água. Nasce nos Andes e deságua no Oceano Atlântico.", "geografia")

print(base.buscar("Qual é o maior rio do mundo?", k=2))            # só a busca (para checar se acha o trecho certo)
print(responder_com_base(base, "Qual é o maior rio do mundo?"))     # busca + resposta do modelo